# Drift model development

Exploration notebook for the DistilBERT sentiment classifier.
Run cells in order. Training writes the model to `backend/app/model/saved_model/`.

In [ ]:
import sys
sys.path.insert(0, '../backend')

## 1. Load and inspect data

In [ ]:
from app.data.loader import load_all
from collections import Counter

records = load_all()
print(f"Total records: {len(records)}")

label_counts = Counter(r['sentiment_label'] for r in records)
label_names = {0: 'negative', 1: 'neutral', 2: 'positive'}
for label, count in sorted(label_counts.items()):
    print(f"  {label_names[label]}: {count} ({count/len(records)*100:.1f}%)")

In [ ]:
# Sample a few records to sanity check
import random
for r in random.sample(records, 5):
    print(f"[{label_names[r['sentiment_label']]}] {r['text'][:100]}")
    print()

## 2. Preprocess

In [ ]:
from app.data.preprocessor import preprocess

train_ds, val_ds, test_ds, tokenizer = preprocess(records)
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

## 3. Train

In [ ]:
from app.model.train import train

metrics = train()
print(metrics)

## 4. Evaluate and inspect errors

In [ ]:
from app.model.evaluate import evaluate

cm_dict = evaluate()

In [ ]:
import json

with open('../backend/app/model/saved_model/misclassified.json') as f:
    misclassified = json.load(f)

for ex in misclassified[:5]:
    print(f"True: {ex['true_label']} | Predicted: {ex['predicted_label']}")
    print(f"Text: {ex['text'][:120]}")
    print()

## 5. Test inference and SHAP

In [ ]:
from app.model.predict import predict_single, explain

test_texts = [
    "This is absolutely amazing, I love it!",
    "Terrible experience, would not recommend.",
    "It was okay, nothing special.",
]

for text in test_texts:
    result = predict_single(text)
    print(f"{result['label']:8} ({result['scores']['positive']:.2f}+) | {text}")

In [ ]:
tokens = explain("This product is absolutely fantastic but the shipping was terrible.")
for t in tokens:
    print(f"{t['token']:20} {t['score']:+.4f}")